In [1]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
import warnings
warnings.filterwarnings("ignore")

print("⏳ KHỞI ĐỘNG ĐỘNG CƠ ĐỊNH GIÁ TRÁI PHIẾU...")

# ====================================================================
# 1. KẾT NỐI HẠ TẦNG BIGQUERY (AUTHENTICATION)
# ====================================================================
project_id = "jda-k1"
dataset_id = "cuongnm_data_pipeline"
key_path = r"C:\Users\DELL\Desktop\Project\Project Market risk\Key - path 2afad6d8f47e.json"

credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=project_id)

print("-> BigQuery thực hiện CROSS JOIN...")

# ====================================================================
# 2. KÉO DỮ LIỆU BẰNG ADVANCED SQL 
# ====================================================================
sql_pull_data = f"""
    -- KỸ NĂNG SQL NÂNG CAO: CROSS JOIN VÀ SUBQUERY
    SELECT 
        f.Date, d.Bond_ID, d.Face_Value, d.Coupon_Rate, d.Maturity_Date, d.Payment_Freq, d.Z_Spread,
        f.Yield_1Y, f.Yield_2Y, f.Yield_3Y, f.Yield_5Y, f.Yield_10Y
    FROM `{project_id}.{dataset_id}.FACT_Yield_Curve` AS f
    CROSS JOIN `{project_id}.{dataset_id}.DIM_Bond_Portfolio` AS d
    -- Lọc ra đúng ngày gần nhất có dữ liệu Lợi suất
    WHERE f.Date = (SELECT MAX(Date) FROM `{project_id}.{dataset_id}.FACT_Yield_Curve`)
"""
df_bond_pricing_today = client.query(sql_pull_data).to_dataframe()

print("✅ Đã kéo thành công Data sạch! Sẵn sàng cho Toán học.")
print("\n[XEM TRƯỚC BẢNG DỮ LIỆU GỘP]:")
print(df_bond_pricing_today.head())

⏳ KHỞI ĐỘNG ĐỘNG CƠ ĐỊNH GIÁ TRÁI PHIẾU...
-> BigQuery thực hiện CROSS JOIN...
✅ Đã kéo thành công Data sạch! Sẵn sàng cho Toán học.

[XEM TRƯỚC BẢNG DỮ LIỆU GỘP]:
         Date  Bond_ID  Face_Value  Coupon_Rate Maturity_Date  Payment_Freq  \
0  2026-07-20  MSN2026  1000000000        0.085    2026-12-31             1   
1  2026-07-20  HPG2028  1000000000        0.078    2028-10-10             1   
2  2026-07-20  TCB2025  1000000000        0.065    2025-12-01             1   
3  2026-07-20  VIC2027  1000000000        0.092    2027-05-15             2   
4  2026-07-20  VHM2029  1000000000        0.095    2029-01-20             2   

   Z_Spread  Yield_1Y  Yield_2Y  Yield_3Y  Yield_5Y  Yield_10Y  
0     0.025  0.044093  0.049215  0.053127  0.059032   0.069116  
1     0.015  0.044093  0.049215  0.053127  0.059032   0.069116  
2     0.012  0.044093  0.049215  0.053127  0.059032   0.069116  
3     0.030  0.044093  0.049215  0.053127  0.059032   0.069116  
4     0.035  0.044093  0.049215  0.0

In [2]:
# ====================================================================
# 3. LÕI TOÁN HỌC: ĐỘNG CƠ CHIẾT KHẤU DÒNG TIỀN (DCF ENGINE)
# ====================================================================
import numpy as np
class BondPricingEngine:
    def __init__(self, df):
        self.df = df.copy()
        self.df['Date'] = pd.to_datetime(self.df['Date'])
        self.df['Maturity_Date'] = pd.to_datetime(self.df['Maturity_Date'])

    def _interpolate_yield(self, t, row):
        """
        Toán học Nội suy (Linear Interpolation): 
        Tính ra Lãi suất phi rủi ro chính xác cho số ngày lẻ của Trái phiếu
        dựa trên các điểm neo 1Y, 2Y, 3Y, 5Y, 10Y.
        """
        y1, y2, y3 = row['Yield_1Y'], row['Yield_2Y'], row['Yield_3Y']
        y5, y10 = row['Yield_5Y'], row['Yield_10Y']
        
        if t <= 1: return y1
        elif t <= 2: return y1 + (y2 - y1) * (t - 1) / (2 - 1)
        elif t <= 3: return y2 + (y3 - y2) * (t - 2) / (3 - 2)
        elif t <= 5: return y3 + (y5 - y3) * (t - 3) / (5 - 3)
        elif t <= 10: return y5 + (y10 - y5) * (t - 5) / (10 - 5)
        else: return y10 # Trái phiếu > 10 năm mặc định lấy mốc 10Y

    def _calculate_bond_metrics(self, row):
        # 1. Tính Thời gian đáo hạn (T - Năm)
        T = (row['Maturity_Date'] - row['Date']).days / 365.0
        if T <= 0:
            return pd.Series([0, 0, 0, 0, 0, row['Face_Value'], 0, 0, 0], 
                             index=['Time_to_Maturity', 'Risk_Free_Rate', 'Discount_Rate', 
                                    'Accrued_Interest', 'Dirty_Price', 'Clean_Price', 
                                    'Mac_Duration', 'Mod_Duration', 'Convexity'])

        # 2. Tìm YTM và Lãi suất từng kỳ
        rf_rate = self._interpolate_yield(T, row)
        r = rf_rate + row['Z_Spread']  
        
        freq = row['Payment_Freq']
        r_period = r / freq
        c_period = row['Coupon_Rate'] / freq * row['Face_Value']
        
        # 3. KỸ THUẬT TÁCH DÒNG TIỀN (FRACTIONAL PERIODS)
        periods_remaining = T * freq
        num_cash_flows = int(np.ceil(periods_remaining)) # Tổng số lần nhận tiền còn lại
        
        # Tìm thời gian đến kỳ nhận lãi GẦN NHẤT (w)
        w = periods_remaining % 1
        if w == 0: 
            w = 1.0 # Nếu hôm nay vừa trả lãi xong, kỳ tới đếm từ 1
            fraction_elapsed = 0.0 # Lãi cộng dồn reset về 0
        else:
            fraction_elapsed = 1.0 - w # Phần trăm thời gian đã trôi qua kể từ kỳ trước
            
        # TÍNH LÃI CỘNG DỒN (AI)
        accrued_interest = fraction_elapsed * c_period

        # 4. DCF TRÊN KỲ HẠN LẺ
        pv = 0         
        pv_t = 0       
        conv_sum = 0   
        
        for i in range(num_cash_flows):
            # Kỳ hạn thực tế của dòng tiền (Ví dụ: 0.8, 1.8, 2.8)
            t_period = w + i 
            t_years = t_period / freq
            df = (1 + r_period) ** t_period  # Hệ số chiết khấu phân số
            
            # Cục tiền kỳ cuối được cộng thêm Mệnh giá
            cf = c_period if i < (num_cash_flows - 1) else c_period + row['Face_Value']
            
            pv_cf = cf / df
            pv += pv_cf
            pv_t += pv_cf * t_years
            conv_sum += (cf * t_period * (t_period + 1)) / ((1 + r_period)**(t_period + 2))

        # 5. CHỐT HẠ BỘ 3 QUYỀN LỰC & CHỈ SỐ RỦI RO
        dirty_price = pv
        clean_price = dirty_price - accrued_interest
        
        mac_duration = pv_t / dirty_price
        mod_duration = mac_duration / (1 + r_period)
        convexity = (conv_sum / dirty_price) / (freq ** 2)

        return pd.Series([T, rf_rate, r, accrued_interest, dirty_price, clean_price, mac_duration, mod_duration, convexity], 
                         index=['Time_to_Maturity', 'Risk_Free_Rate', 'Discount_Rate', 
                                'Accrued_Interest', 'Dirty_Price', 'Clean_Price', 
                                'Mac_Duration', 'Mod_Duration', 'Convexity'])
        
    def run_engine(self):
        # Áp dụng thuật toán cho từng dòng Trái phiếu trong Sổ cái
        metrics_df = self.df.apply(self._calculate_bond_metrics, axis=1)
        
        # Ghép kết quả toán học vào bảng dữ liệu gốc
        self.df = pd.concat([self.df, metrics_df], axis=1)
        
        # Format lại một vài cột cho đẹp
        self.df['Clean_Price'] = self.df['Clean_Price'].round(2)
        self.df['Dirty_Price'] = self.df['Dirty_Price'].round(2)
        self.df['Mod_Duration'] = self.df['Mod_Duration'].round(4)
        return self.df

# ====================================================================
# BÓP CÒ ĐỘNG CƠ
# ====================================================================
print("-> Đang đưa dữ liệu vào Động cơ Toán học...")
engine = BondPricingEngine(df_bond_pricing_today)
df_final_results = engine.run_engine()

print("✅ TOÁN HỌC HOÀN TẤT! XEM TRƯỚC BÁO CÁO ĐỊNH GIÁ:")
# In ra các cột quan trọng nhất 
columns_to_show = ['Bond_ID', 'Time_to_Maturity', 'Discount_Rate', 'Clean_Price', 'Mod_Duration', 'Convexity']
print(df_final_results[columns_to_show])

-> Đang đưa dữ liệu vào Động cơ Toán học...
✅ TOÁN HỌC HOÀN TẤT! XEM TRƯỚC BÁO CÁO ĐỊNH GIÁ:
   Bond_ID  Time_to_Maturity  Discount_Rate   Clean_Price  Mod_Duration  \
0  MSN2026          0.449315       0.069093  1.006105e+09        0.4203   
1  HPG2028          2.227397       0.065105  1.025526e+09        1.8959   
2  TCB2025          0.000000       0.000000  1.000000e+09        0.0000   
3  VIC2027          0.819178       0.074093  1.013790e+09        0.7689   
4  VHM2029          2.506849       0.086198  1.019457e+09        2.1003   

   Convexity  
0   0.569748  
1   5.644347  
2   0.000000  
3   0.971590  
4   5.867969  


In [3]:
# ====================================================================
# 4. MODULE STRESS TEST: SỐC LÃI SUẤT & DELTA EVE (IRRBB)
# ====================================================================
def run_stress_test(df_portfolio, shock_bps=100):
    """
    Thuật toán lượng hóa thiệt hại danh mục khi Lãi suất giật cục.
    Mặc định: Tăng 100 điểm cơ bản (+1%).
    """
    df_stress = df_portfolio.copy()
    
    # Đổi điểm cơ bản (bps) ra số thập phân (100 bps = 0.01)
    shock_rate = shock_bps / 10000.0
    
    # Áp dụng phương trình Chuỗi Taylor bậc 2
    df_stress['Price_Change_Pct'] = -df_stress['Mod_Duration'] * shock_rate + 0.5 * df_stress['Convexity'] * (shock_rate ** 2)
    
    # Tính ra số tiền thiệt hại tuyệt đối (P&L Impact) trên mỗi Trái phiếu
    df_stress['PnL_Impact_VND'] = df_stress['Clean_Price'] * df_stress['Price_Change_Pct']
    
    return df_stress

print("-> NHNN ra tin tăng lãi suất 1% (+100 bps)!")
print("-> Đang chạy kịch bản Stress Test...")

df_stressed = run_stress_test(df_final_results, shock_bps=100)

columns_to_show = ['Bond_ID', 'Time_to_Maturity', 'Mod_Duration', 'Price_Change_Pct', 'PnL_Impact_VND']

print("\n✅ BÁO CÁO THIỆT HẠI DANH MỤC (STRESS TEST REPORT):")
# Nhân 100 để hiển thị dạng % cho đẹp
df_display = df_stressed[columns_to_show].copy()
df_display['Price_Change_Pct'] = (df_display['Price_Change_Pct'] * 100).round(2).astype(str) + '%'
print(df_display)

# Tổng thiệt hại của toàn bộ Sổ ngân hàng (Delta EVE)
delta_eve = df_stressed['PnL_Impact_VND'].sum()
print("="*75)
print(f"🚨 TỔNG THIỆT HẠI TOÀN DANH MỤC (DELTA EVE): {delta_eve:,.2f} VND")
print("="*75)

-> NHNN ra tin tăng lãi suất 1% (+100 bps)!
-> Đang chạy kịch bản Stress Test...

✅ BÁO CÁO THIỆT HẠI DANH MỤC (STRESS TEST REPORT):
   Bond_ID  Time_to_Maturity  Mod_Duration Price_Change_Pct  PnL_Impact_VND
0  MSN2026          0.449315        0.4203           -0.42%   -4.199998e+06
1  HPG2028          2.227397        1.8959           -1.87%   -1.915352e+07
2  TCB2025          0.000000        0.0000             0.0%    0.000000e+00
3  VIC2027          0.819178        0.7689           -0.76%   -7.745781e+06
4  VHM2029          2.506849        2.1003           -2.07%   -2.111256e+07
🚨 TỔNG THIỆT HẠI TOÀN DANH MỤC (DELTA EVE): -52,211,855.24 VND


In [4]:
# ====================================================================
# 5. THE RETURN TRIP: LƯU BÁO CÁO RỦI RO LÊN DATA WAREHOUSE
# ====================================================================
print("-> Bắt đầu khởi hành Chuyến tàu khứ hồi...")

table_report_id = f"{project_id}.{dataset_id}.Report_Bond_Pricing_Risk"

# Dùng lệnh WRITE_TRUNCATE để ghi đè báo cáo mới nhất của ngày hôm nay
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Bắn DataFrame chứa kết quả Stress Test (df_stressed) lên BigQuery
client.load_table_from_dataframe(df_stressed, table_report_id, job_config=job_config).result()

print(f"✅ XUẤT SẮC! Báo cáo Rủi ro đã được đóng gói và chuyển lên kho: {table_report_id}")
print("-> HỆ THỐNG PYTHON END-TO-END ĐÃ HOÀN TẤT!")

-> Bắt đầu khởi hành Chuyến tàu khứ hồi...
✅ XUẤT SẮC! Báo cáo Rủi ro đã được đóng gói và chuyển lên kho: jda-k1.cuongnm_data_pipeline.Report_Bond_Pricing_Risk
-> HỆ THỐNG PYTHON END-TO-END ĐÃ HOÀN TẤT!
